# Configuração de ambiente

In [ ]:
# !unzip mestrado_ifes.zip
# !mv mestrado_ifes-jules_wip_13570567048251198718/* .
# !rm -R mestrado_ifes-jules_wip_13570567048251198718

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ! mkdir ~/.kaggle
# ! cp kaggle.json ~/.kaggle/
# ! chmod 600 ~/.kaggle/kaggle.json
# !kaggle competitions download -c isic-2024-challenge
# !unzip 'drive/MyDrive/Acadêmico/IFES/01 - Dissertação/Dados/isic-2024-challenge.zip' -d Image
!mkdir '/content/Image'
!cp -r 'drive/MyDrive/Acadêmico/IFES/01 - Dissertação/Code/Image_Segmentation' Image_Segmentation
!cp 'drive/MyDrive/Acadêmico/IFES/01 - Dissertação/Code/U_Net-100-0.0005-70-0.6667.pkl' U_Net-100-0.0005-70-0.6667.pkl
!cp -r 'drive/MyDrive/Acadêmico/IFES/01 - Dissertação/Dados/sample-images' '/content/Image/sample-image'
!cp -r 'drive/MyDrive/Acadêmico/IFES/01 - Dissertação/Dados/sample-metadata.csv' '/content/Image/sample-metadata.csv'
!cp 'drive/MyDrive/Acadêmico/IFES/01 - Dissertação/Dados/RAG.zip' '/content/RAG.zip'

In [ ]:
#%pip install crewai crewai-tools poetry vllm
!curl -fsSL https://ollama.com/install.sh | sh
!pip install pymupdf langchain-community langchain-core langgraph faiss-cpu langchain_openai
!pip install langchain --upgrade

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading Linux amd64 bundle
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [ ]:
!nohup ollama serve > ollama.log &
!nohup ollama run llava:13b &

nohup: redirecting stderr to stdout
nohup: appending output to 'nohup.out'


In [ ]:
import pymupdf  # PyMuPDF
import os
import zipfile
from langchain_community.chat_models import ChatOllama
from langchain_openai import ChatOpenAI
from langchain.schema import SystemMessage, HumanMessage
from langchain_community.vectorstores.faiss import FAISS
from langchain.embeddings import SentenceTransformerEmbeddings
from langchain.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.chains import RetrievalQA
from langgraph.graph import StateGraph, START, END
from langchain.output_parsers import PydanticOutputParser
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain.prompts import PromptTemplate


from pydantic import BaseModel, Field
from typing import Union
import numpy as np

# Criação da RAG leve com base no PDF

In [ ]:
# Caminho do arquivo ZIP enviado pelo usuário
zip_path = "RAG.zip"
extracted_path = "extracted_pdfs"

# Extraindo os PDFs
os.makedirs(extracted_path, exist_ok=True)
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extracted_path)

# Função para extrair texto de PDFs
def extract_text_from_pdfs(pdf_folder):
    text_data = ""
    for pdf_file in os.listdir(pdf_folder):
        if pdf_file.endswith(".pdf"):
            pdf_path = os.path.join(pdf_folder, pdf_file)
            doc = pymupdf.open(pdf_path)
            for page in doc:
                text_data += page.get_text("text") + "\n\n"
    return text_data

# Extraindo texto dos PDFs
medical_texts = extract_text_from_pdfs(extracted_path)

# Salvando em um arquivo de texto
medical_texts_path = "medical_texts.txt"
with open(medical_texts_path, "w", encoding="utf-8") as f:
    f.write(medical_texts)

# Retornar o caminho do arquivo salvo
medical_texts_path


'medical_texts.txt'

# Criando objeto do RAG

In [ ]:
# Carregar embeddings do Sentence-Transformers
embeddings = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")

# Carregar e processar documentos médicos (já extraídos dos PDFs)
loader = TextLoader("medical_texts.txt")  # Arquivo consolidado com informações médicas extraídas
texts = loader.load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
documents = text_splitter.split_documents(texts)

# Criar banco vetorial FAISS
vectorstore = FAISS.from_documents(documents, embeddings)
retriever = vectorstore.as_retriever()

# Leitura do modelo de segmentação selecionado

In [ ]:
from Image_Segmentation.code.network import U_Net,R2U_Net,AttU_Net,R2AttU_Net
import torch
import torch.nn.functional as F
from torchvision import transforms as T
import random
import base64
from PIL import Image
from io import BytesIO


model = U_Net(img_ch=3,output_ch=1)
model.load_state_dict(torch.load('./U_Net-100-0.0005-70-0.6667.pkl', map_location=torch.device('cpu')))

<All keys matched successfully>

# Criação dos agentes e fluxo

In [ ]:
# Configurar modelo LLM
# model_name = "llava:13b"
# llm = ChatOllama(model="llava:13b")

# openai_api_key = "dummy" # Replace with your actual API key
model_name="gpt-4o-mini"
# llm = ChatOpenAI(openai_api_key=openai_api_key, model=model_name, temperature=0)


In [ ]:
class ABCDDiagnosticAnswer(BaseModel):
  assimetria: str = Field(description="Valor de 0 a 2 e motivo.")
  bordas: str = Field(description="Valor de 0 a 8 e motivo.")
  cor: str = Field(description="Valor de 1 a 6 e motivo.")
  estruturas: str = Field(description="Valor de 1 a 5 e motivo.")
  explicacao: str = Field(description="Explicação das pontuações dadas.")

class MenziesPositiveAnswer(BaseModel):
  veu_azul_branco: str = Field(description="Valor 0 ou 1. Motivo.")
  multiplos_pontos_marrons: str = Field(description="Valor 0 ou 1. Motivo.")
  pseudopodes: str = Field(description="Valor 0 ou 1. Motivo.")
  streaming_radial: str = Field(description="Valor 0 ou 1. Motivo.")
  despigmentacao_tipo_cicatriz: str = Field(description="Valor 0 ou 1. Motivo.")
  pontos_pretos_perifericos: str = Field(description="Valor 0 ou 1. Motivo.")
  multiplas_cores: str = Field(description="Valor 0 ou 1. Motivo")

class MenziesNegativeAnswer(BaseModel):
  simetria: str = Field(description="Valor 0 ou 1. Motivo.")
  cor_unica: str = Field(description="Valor 0 ou 1. Motivo.")

class MenziesDiagnosticAnswer(BaseModel):
  positivas: MenziesPositiveAnswer = Field(description="Pontos que positivam o diagnóstico")
  negativas: MenziesNegativeAnswer = Field(description="Pontos que negativam o diagnóstico.")
  explicacao: str = Field(description="Explicação das pontuações dadas.")

class SPCLDiagnosticAnswer(BaseModel):
  padrao_de_pigmentacao_atipico: str = Field(description="Sim ou não e motivo.")
  veu_azul_branco_irregular: str = Field(description="Sim ou não e motivo.")
  vasos_atipicos: str = Field(description="Sim ou não e motivo.")
  padrao_de_reticulo_irregular: str = Field(description="Sim ou não e motivo.")
  padrao_de_globulos_irregulares: str = Field(description="Sim ou não e motivo.")
  hiperpigmentacao_localizada: str = Field(description="Sim ou não e motivo.")
  regressao: str = Field(description="Sim ou não e motivo.")
  explicacao: str = Field(description="Explicação das pontuações dadas.")

class SummaryAnswer(BaseModel):
  diagnostico: str = Field(description="Suspeita de Melanoma / Lesão Benigna / Lesão Indeterminada")
  justificativa: str = Field(description="Integração das evidências dos algoritmos ABCD, Menzies e SPCL, destacando os principais achados.")
  recomendacoes: str = Field(description="Sugestões clínicas como biópsia, acompanhamento, ou nenhuma ação imediata.")

In [ ]:
# Configurar agentes
class SegmentationAgent:
  def __init__(self,model,threshold=.5,image_size=224,mode='train',augmentation_prob=0.4):
    self.model = model
    self.image_size = image_size
    self.mode = mode
    self.RotationDegree = [0,90,180,270]
    self.augmentation_prob = augmentation_prob

  def convert_to_base64(self, pil_image):
    """
    Convert PIL images to Base64 encoded strings

    :param pil_image: PIL image
    :return: Base64 string
    """
    pil_image = pil_image.convert("RGB")
    buffered = BytesIO()
    pil_image.save(buffered, format="JPEG")
    img_str = base64.b64encode(buffered.getvalue()).decode("utf-8")
    return img_str

  def segment_image(self, state):
    """Reads an image from a file and preprocesses it and returns."""
    img = Image.open(state.image_path)
    encoded_string = self.convert_to_base64(img)

    aspect_ratio = img.size[1]/img.size[0]

    Transform = []

    ResizeRange = random.randint(300,320)
    Transform.append(T.Resize((int(ResizeRange*aspect_ratio),ResizeRange)))
    p_transform = random.random()

    Transform.append(T.Resize((int(256*aspect_ratio)-int(256*aspect_ratio)%16,256)))
    Transform.append(T.ToTensor())
    Transform = T.Compose(Transform)
    img_ = img
    img = Transform(img)

    SR = model(img.unsqueeze(0))
    SR_s = F.sigmoid(SR)

    SR_image = (SR_s>.4)*img
    im = T.ToPILImage()(SR_image[0])
    seg_encoded_string = self.convert_to_base64(im)

    return {"seg_image_data": seg_encoded_string, "image_data": encoded_string}


class ABCDDiagnosticAgent:
    def __init__(self, model_name, output_format):
        if model_name == "llava:13b":
          self.llm = ChatOllama(model=model_name)
        else:
          openai_api_key = "dummy" # Replace with your actual API key
          self.llm = ChatOpenAI(openai_api_key=openai_api_key, model=model_name, temperature=.7)
        self.parser = PydanticOutputParser(pydantic_object=output_format)

    def analyze_lesion(self, state):
      content_str = """
        <-- Identidade do Agente -->
        Você está atuando como um sistema de apoio à pesquisa em análise de imagens dermatológicas.

        Sua função é identificar e descrever padrões visuais relevantes encontrados nas imagens fornecidas, utilizando critérios técnicos que serão informados abaixo.

        Não forneça um diagnóstico ou julgamento clínico. Apenas descreva o que é visualmente observado com base em critérios objetivos.

        <-- Instruções Gerais -->
        - Ignore variações de cor típicas da pele humana, como tons naturais da derme ou pequenas sombras.
        - Só considere estruturas dermatoscópicas se forem claramente identificáveis, sem dúvida razoável.
        - Não atribua significância clínica a artefatos visuais ou ruídos.
        - Seja conservador ao identificar padrões visuais suspeitos. Se a evidência não for forte, trate como característica benigna ou neutra.

        Você recebe uma imagem para análise:
        Imagem segmentada, que realça os contornos e características internas da lesão. {image_segmentada}

        <-- Instruções Específicas -->

        Avalie a lesão cutânea na imagem abaixo utilizando o algoritmo ABCD de dermoscopia. Para cada um dos critérios (Assimetria, Bordas, Cor e Estruturas Dermoscópicas), forneça a pontuação de acordo com a seguinte escala:
        Critérios Gerais:
          Considere assimetrias somente aquelas mais acentuadas, caso seja leve, não considere.
          Ao avaliar as cores, desconsidere da contagem a cor da pele do indivíduo para não enviesar a análise
        Assimetria (A): Considere assimetrias somente aquelas acentuadas, caso seja leve, não considere.
          0: Nenhuma assimetria
          1: Assimetria em um eixo
          2: Assimetria em ambos os eixos
        Bordas (B): Divida a imagem em 8 quadrantes de ângulos iguais. Dentro de cada quadrante avalie se a borda tem fim claro delimitado ou não.
          0: Bordas indistintas em todos os quadrantes
          1-8: Bordas nítidas em alguns ou todos os quadrantes (pontuação proporcional)
        Cor (C): Ao avaliar as cores, desconsidere da contagem a cor da pele do indivíduo para não enviesar a análise
          Atribua 1 ponto para cada cor presente (branco/bege, vermelho/rosa, marrom claro, marrom escuro, azul-cinza, preto).
        Estruturas Dermoscópicas (D):
          Atribua 1 ponto para cada estrutura observada (áreas sem estrutura, rede pigmentada, linhas ramificadas, pontos, glóbulos).
        Resultado final:
          Faça a soma ponderada dos resultados de cada critério, com pesos de 1.3, .1, .5 e .5, respectivamente. Mostre o cálculo da soma ponderada passo a passo. Caso o valor seja inferior a 4.75, é uma lesão benigna. Caso seja superior a 4.75, mas inferior a 5.45 é uma lesão suspeita. Caso seja maior que 5.45 é uma lesão maligna.
        Forneça a resposta no formato JSON:
q      """

      prompt = PromptTemplate(
          template=content_str,
          input_variables=["image_original", "image_segmentada"],
          partial_variables={"format_instructions": self.parser.get_format_instructions()},
      )
      chain = prompt | self.llm
      if model_name != "llava:13b":
          chain = chain | self.parser
      img1 = f"data:image/jpeg;base64,{state.image_data}"
      img2 = f"data:image/jpeg;base64,{state.seg_image_data}"
      output = chain.invoke({"image_original": img1, "image_segmentada": img2})
      if model_name == "llava:13b":
        output = output.content

      return {"diagnosis_abcd": output}


class MenziesDiagnosticAgent:
    def __init__(self, model_name, output_format):
        if model_name == "llava:13b":
          self.llm = ChatOllama(model=model_name)
        else:
          openai_api_key = "dummy" # Replace with your actual API key
          self.llm = ChatOpenAI(openai_api_key=openai_api_key, model=model_name, temperature=.7)
        self.parser = PydanticOutputParser(pydantic_object=output_format)

    def analyze_lesion(self, state):
      content_str = """
      <-- Identidade do Agente -->
      Você está atuando como um sistema de apoio à pesquisa em análise de imagens dermatológicas.

      Sua função é identificar e descrever padrões visuais relevantes encontrados nas imagens fornecidas, utilizando critérios técnicos que serão informados abaixo.

      Não forneça um diagnóstico ou julgamento clínico. Apenas descreva o que é visualmente observado com base em critérios objetivos.

      <-- Instruções Gerais -->
      - Ignore variações de cor típicas da pele humana, como tons naturais da derme ou pequenas sombras.
      - Só considere estruturas dermatoscópicas se forem claramente identificáveis, sem dúvida razoável.
      - Não atribua significância clínica a artefatos visuais ou ruídos.
      - Seja conservador ao identificar padrões visuais suspeitos. Se a evidência não for forte, trate como característica benigna ou neutra.
      - Considere assimetrias somente aquelas acentuadas, caso seja leve, não considere.

      Você recebe uma imagem para análise:
        Imagem segmentada, que realça os contornos e características internas da lesão. {image_segmentada}

      <-- Instruções Específicas -->
      Avalie a lesão cutânea na imagem abaixo utilizando o Método Menzies. Aplique os critérios para as Características Positivas e Características Negativas da seguinte forma:

      Características Positivas (pelo menos uma deve estar presente para diagnóstico de melanoma):
        Véu Azul-Branco: Se presente, pontue 1. Caso contrário, pontue 0.
        Múltiplos Pontos Marrons: Se presentes, pontue 1. Caso contrário, pontue 0.
        Pseudópodes: Se presentes, pontue 1. Caso contrário, pontue 0.
        Streaming Radial: Se presente, pontue 1. Caso contrário, pontue 0.
        Despigmentação Tipo Cicatriz: Se presente, pontue 1. Caso contrário, pontue 0.
        Pontos/Glóbulos Pretos Periféricos: Se presentes, pontue 1. Caso contrário, pontue 0.
        Múltiplas Cores (vermelho/rosa, branco/bege, marrom escuro, preto, cinza e azul): Se cinco ou seis cores presentes, pontue 1. Caso contrário, pontue 0.
        Vários pontos/saliências azul-acinzentados: Se presentes, pontue 1. Caso contrário, pontue 0.
        Linhas ramificadas: Se presentes, pontue 1. Caso contrário, pontue 0.
      Características Negativas (ambas devem estar ausentes para diagnóstico de melanoma):
        Cor Única: Se a lesão apresentar apenas uma cor, pontue 1. Caso contrário, pontue 0.
        Simetria do padrão de pigmentação: Se a lesão for simétrica na distribuição de cores, pontue 1. Caso contrário, pontue 0.

      Resultado Final:
        A lesão será classificada como melanoma caso possua 1 característica positiva e as 2 características negativas estiverem ausentes. Se houver apenas 1 característica negativa presente, indica suspeita de melanoma. Não havendo caracterísitica positiva e tendo as 2 características negativas presentes, caracteriza lesão benigna.
      Forneça a resposta no formato JSON:
      {format_instructions}
      """
      prompt = PromptTemplate(
          template=content_str,
          input_variables=["image_original", "image_segmentada"],
          partial_variables={"format_instructions": self.parser.get_format_instructions()},
      )
      chain = prompt | self.llm
      if model_name != "llava:13b":
          chain = chain | self.parser
      img1 = f"data:image/jpeg;base64,{state.image_data}"
      img2 = f"data:image/jpeg;base64,{state.seg_image_data}"
      output = chain.invoke({"image_original": img1, "image_segmentada": img2})
      if model_name == "llava:13b":
        output = output.content

      return {"diagnosis_menzies": output}


class SPCLDiagnosticAgent:
    def __init__(self, model_name, output_format):
        if model_name == "llava:13b":
          self.llm = ChatOllama(model=model_name)
        else:
          openai_api_key = "dummy" # Replace with your actual API key
          self.llm = ChatOpenAI(openai_api_key=openai_api_key, model=model_name, temperature=.7)
        self.parser = PydanticOutputParser(pydantic_object=output_format)

    def analyze_lesion(self, state):
      content_str = """
      <-- Identidade do Agente -->
      Você está atuando como um sistema de apoio à pesquisa em análise de imagens dermatológicas.

      Sua função é identificar e descrever padrões visuais relevantes encontrados nas imagens fornecidas, utilizando critérios técnicos que serão informados abaixo.

      Não forneça um diagnóstico ou julgamento clínico. Apenas descreva o que é visualmente observado com base em critérios objetivos.
      <-- Instruções Gerais -->
      - Ignore variações de cor típicas da pele humana, como tons naturais da derme ou pequenas sombras.
      - Só considere estruturas dermatoscópicas se forem claramente identificáveis, sem dúvida razoável.
      - Não atribua significância clínica a artefatos visuais ou ruídos.
      - Seja conservador ao identificar padrões visuais suspeitos. Se a evidência não for forte, trate como característica benigna ou neutra.
      - Considere assimetrias somente aquelas mais acentuadas, caso seja leve, não considere.
      - Não considerar critério de bordas, diâmetro e sangramento na análise.

      Você recebe uma imagem para análise:
        Imagem segmentada, que realça os contornos e características internas da lesão. {image_segmentada}

      <-- Instruções Específicas -->

      Avalie a lesão cutânea na imagem abaixo utilizando a Checklist de Sete Pontos (Seven-Point Checklist). Aplique os critérios conforme descrito abaixo:
      Critérios Maiores (2 pontos cada):
        Padrão de pigmentação atípico: Se presente, pontue 2. Caso contrário, pontue 0.
        Véu azul-branco irregular: Se presente, pontue 2. Caso contrário, pontue 0.
        Vasos atípicos: Se presentes, pontue 2. Caso contrário, pontue 0.
      Critérios Menores (1 ponto cada):
        Padrão de retículo irregular: Se presente, pontue 1. Caso contrário, pontue 0.
        Padrão de glóbulos irregulares: Se presente, pontue 1. Caso contrário, pontue 0.  Considere irregularidades somente aquelas mais acentuadas, caso seja leve, não considere.
        Hiperpigmentação localizada (pontos escuros localizados): Se presente, pontue 1. Caso contrário, pontue 0.
        Regressão (áreas esbranquiçadas ou azul-acinzentadas): Se presente, pontue 1. Caso contrário, pontue 0.
      Resultado Final:
        A lesão será considerada suspeita para melanoma se a pontuação total for 3 ou mais pontos.
        Pontuações inferiores a 3 indicam menor probabilidade de malignidade, mas não descartam a necessidade de avaliação clínica.

      Retorne uma resposta no formato JSON, baseado nas observações visuais de ambas imagens
      fornecidas:
      {format_instructions}
      """
      prompt = PromptTemplate(
          template=content_str,
          input_variables=["image_original", "image_segmentada"],
          partial_variables={"format_instructions": self.parser.get_format_instructions()},
      )
      chain = prompt | self.llm
      if model_name != "llava:13b":
          chain = chain | self.parser
      img1 = f"data:image/jpeg;base64,{state.image_data}"
      img2 = f"data:image/jpeg;base64,{state.seg_image_data}"
      output = chain.invoke({"image_original": img1, "image_segmentada": img2})
      if model_name == "llava:13b":
        output = output.content

      return {"diagnosis_spcl": output}

class SummaryAgent:
    def __init__(self, model_name, output_format):
        if model_name == "llava:13b":
          self.llm = ChatOllama(model=model_name)
        else:
          openai_api_key = "dummy" # Replace with your actual API key
          self.llm = ChatOpenAI(openai_api_key=openai_api_key, model=model_name, temperature=.7)
        self.parser = PydanticOutputParser(pydantic_object=output_format)

    def summarize(self, state):
      content_str="""
      <-- Identidade do Agente -->
      Você está atuando como um sistema de apoio à pesquisa em análise de imagens dermatológicas.
      Não forneça um diagnóstico ou julgamento clínico.


      Você recebe uma imagem para análise:
        Imagem segmentada, que realça os contornos e características internas da lesão. {image_segmentada}

      <-- Instruções Específicas -->
      Você está atuando como um especialista que recebeu três pareceres técnicos distintos sobre uma mesma lesão cutânea, baseados nos seguintes algoritmos:
      1️⃣ ABCD – {diagnosis_abcd}
      2️⃣ Menzies – {diagnosis_menzies}
      3️⃣ SPCL (Seven Point Checklist) – {diagnosis_spcl}

      Sua função é integrar os três pareceres fornecidos, comparando seus resultados e justificativas, para construir um **diagnóstico consolidado e fundamentado**.

      Considere:
      - O grau de concordância ou conflito entre os algoritmos;
      - A presença de padrões de alto risco recorrentes nos pareceres;
      - A confiabilidade dos critérios observados (por exemplo, múltiplas cores, bordas irregulares e estruturas atípicas);
      - A gravidade potencial com base em critérios combinados.
      - Caso haja divergências significativas entre os pareceres, utilize a imagem disponibilizada para avaliar e decidir qual está mais condizente com a lesão

      🧠 Ao final, forneça um parecer clínico estruturado em formato JSON:
      {format_instructions}
      """

      prompt = PromptTemplate(
          template=content_str,
          input_variables=["diagnosis_abcd", "diagnosis_menzies", "diagnosis_spcl"],
          partial_variables={"format_instructions": self.parser.get_format_instructions()},
      )
      chain = prompt | self.llm
      if model_name != "llava:13b":
          chain = chain | self.parser
      output = chain.invoke({"diagnosis_abcd": state.diagnosis_abcd, "diagnosis_menzies": state.diagnosis_menzies, "diagnosis_spcl": state.diagnosis_spcl})
      if model_name == "llava:13b":
        output = output.content

      return {"validation": output}

# Configurar agente crítico
class CriticalReviewAgent:
    def __init__(self, retriever, model_name):
        self.retriever = retriever
        # openai_api_key = "dummy" # Replace with your actual API key
        # self.llm = ChatOpenAI(openai_api_key=openai_api_key, model=model_name, temperature=0)

    def validate_diagnosis(self, state):
        # retrieved_docs = self.retriever.invoke(str(state.validation.model_dump()))
        if model_name == "llava:13b":
          val_res = state.validation
        else:
          val_res = state.validation.model_dump()

        retrieved_docs = self.retriever.invoke(str(val_res))
        return {"final_report": f"Confirmação baseada em literatura médica: {retrieved_docs[0].page_content[:200]}..."}


# Definir o esquema de estado inicial
class GraphState(BaseModel):
    image_path: str
    lesion_size: float
    image_data: str = None
    seg_image_data: str = None
    diagnosis_abcd: Union[str, ABCDDiagnosticAnswer] = None
    diagnosis_menzies: Union[str, MenziesDiagnosticAnswer] = None
    diagnosis_spcl: Union[str, SPCLDiagnosticAnswer] = None
    validation: Union[str, SummaryAnswer] = None
    final_report: str = None

# Criar fluxo no LangGraph
graph = StateGraph(GraphState)

graph.add_node("segmentation", SegmentationAgent(model).segment_image)
graph.add_node("diagnostic_abcd", ABCDDiagnosticAgent(model_name, ABCDDiagnosticAnswer).analyze_lesion)
graph.add_node("diagnostic_menzies", MenziesDiagnosticAgent(model_name, MenziesDiagnosticAnswer).analyze_lesion)
graph.add_node("diagnostic_spcl", SPCLDiagnosticAgent(model_name, SPCLDiagnosticAnswer).analyze_lesion)
graph.add_node("summary", SummaryAgent(model_name, SummaryAnswer).summarize)
graph.add_node("critical_review", CriticalReviewAgent(retriever, model_name).validate_diagnosis)

# Definir conexões do fluxo
graph.add_edge(START, "segmentation")
graph.add_edge("segmentation", "diagnostic_abcd")
graph.add_edge("segmentation", "diagnostic_menzies")
graph.add_edge("segmentation", "diagnostic_spcl")
graph.add_edge(["diagnostic_abcd", "diagnostic_menzies", "diagnostic_spcl"], "summary")
graph.add_edge("summary", "critical_review")
graph.add_edge("critical_review", END)

g = graph.compile()

# Execução do agente

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split


test_df = pd.read_csv('/content/Image/sample-metadata.csv')
# train_df, test_df = train_test_split(
#     dt_ref, test_size=10000, stratify=dt_ref['target'], random_state=42
# )

# target_0_cases = dt_ref[dt_ref['target'] == 0].sample(n=50)
# target_1_cases = dt_ref[dt_ref['target'] == 1].sample(n=50)

# test_df = pd.concat([target_0_cases, target_1_cases])

In [ ]:
test_df.shape

(100, 55)

In [ ]:
!ollama pull llava:13b

In [ ]:
import os
import json
import time
import datetime

shuffled_df = test_df.sample(frac=1, random_state=42).reset_index(drop=True)
list_image = ['/content/Image/sample-image/image/'+i+'.jpg' for i in shuffled_df.isic_id.values]
result_dict = dict()
for j, path in enumerate(list_image):
  if j<10:
    start_time = time.time()
    print(f'{j+1}ª imagem iniciou!')
    i = path.split('/')[-1].replace('.jpg', '')
    for event in g.stream({"image_path": path, 'lesion_size': test_df[test_df['isic_id'] == i]['clin_size_long_diam_mm'].values[0]}, stream_mode="values"):
      for e in ['image_data', 'seg_image_data']:
        event.pop(e, None)
      event['real_value'] = int(test_df[test_df['isic_id'] == i]['target'].values[0])
      result_dict[path] = event
    result_dict[path]['diagnosis_abcd'] = result_dict[path]['diagnosis_abcd'].model_dump()
    result_dict[path]['diagnosis_menzies'] = result_dict[path]['diagnosis_menzies'].model_dump()
    result_dict[path]['diagnosis_spcl'] = result_dict[path]['diagnosis_spcl'].model_dump()
    result_dict[path]['validation'] = result_dict[path]['validation'].model_dump()
    with open(f"drive/MyDrive/Acadêmico/IFES/01 - Dissertação/Dados/llm_result_{model_name}_{datetime.date.today().strftime('%Y%m%d')}.json", "w", encoding="utf-8") as arquivo:
      json.dump(result_dict, arquivo, indent=4, ensure_ascii=False)
    elapsed_time = time.time() - start_time
    print(f"Tempo de análise da {j+1}ª imagem: {elapsed_time:.4f} segundos")

1ª imagem iniciou!
Tempo de análise da 1ª imagem: 18.1748 segundos
2ª imagem iniciou!
Tempo de análise da 2ª imagem: 15.2929 segundos
3ª imagem iniciou!
Tempo de análise da 3ª imagem: 15.2109 segundos
4ª imagem iniciou!
Tempo de análise da 4ª imagem: 13.7886 segundos
5ª imagem iniciou!
Tempo de análise da 5ª imagem: 13.2137 segundos
6ª imagem iniciou!
Tempo de análise da 6ª imagem: 21.1573 segundos
7ª imagem iniciou!
Tempo de análise da 7ª imagem: 12.6651 segundos
8ª imagem iniciou!
Tempo de análise da 8ª imagem: 13.9584 segundos
9ª imagem iniciou!
Tempo de análise da 9ª imagem: 13.5597 segundos
10ª imagem iniciou!
Tempo de análise da 10ª imagem: 12.1769 segundos


In [ ]:
result_dict

{'/content/Image/sample-image/image/ISIC_6767172.jpg': {'image_path': '/content/Image/sample-image/image/ISIC_6767172.jpg',
  'lesion_size': np.float64(2.53),
  'diagnosis_abcd': ' Para realizar a análise da lesão cutânea utilizando o algoritmo ABCD de dermoscopia, primeiro é necessário analisar cada um dos critérios (Assimetria, Bordas, Cor e Estruturas Dermoscópicas) e atribuir pontuações conforme descrito na escala da imagem fornecida:\n\n1. Critério A - Assimetria:\n   Não há assimetrias acentuadas visíveis na imagem, portanto a pontuação para este critério é 0.\n\n2. Critério B - Bordas:\n   Analisando os bordes dentro de cada quadrante, não há bordas nítidas delimitadas, apenas bordas indistintas em todos os quadrantes, então a pontuação para este critério é 0.\n\n3. Critério C - Cor:\n   Considerando que a cor da pele do paciente deve ser desconsiderada ao avaliar as cores, não há necessidade de atribuir pontos nesta etapa. Portanto, a pontuação para este critério é 0.\n\n4. Cri

In [ ]:
i="ISIC_9972649"
dict_state = {
    "image_path": f'/content/Image/train-image/image/{i}.jpg',
    "lesion_size": dt_ref[dt_ref['isic_id'] == i]['clin_size_long_diam_mm'].values[0]
}
for event in g.stream(dict_state, stream_mode="values"):
  pass

In [ ]:
# prompt: Faça um pretty print do result_dict com respeito ao encoding

import json

print(json.dumps(result_dict, indent=4, ensure_ascii=False))


In [ ]:
print("Diagnóstico ABCD:\n\t", event['diagnosis_abcd'],'\n', 100*"=",'\n')
# print("Diagnóstico por Padrão:\n\t", event['diagnosis_pattern'],'\n', 100*"=",'\n')
print("Diagnóstico Menzies:\n\t", event['diagnosis_menzies'],'\n', 100*"=",'\n')
print("Diagnóstico:\n\t", event['diagnosis_spcl'],'\n', 100*"=",'\n')
#print("Diagnóstico:\n\t", event['diagnosis_wspcl'],'\n', 100*"=",'\n')
# print("Prognóstico:\n\t", event['validation'],'\n', 100*"=",'\n')
#print("Relatório:\n\t", event['validation'])

In [ ]:
print(event['diagnosis_menzies'] )

In [ ]:
import json
b = json.loads(event['diagnosis_menzies'].replace("```json", "").replace("```", ""))

In [ ]:
# Score ABCD
a = json.loads(event['diagnosis_abcd'].replace("```json", "").replace("```", ""))
sum_product = sum(x * y for x, y in zip(a['Pontos'].values(), [1.3, .1, .5, .5]))

In [ ]:
sum_product

# Testes de desenvolvimento

In [ ]:
im = T.ToPILImage()(event['im_teste'][3][0])
im

In [ ]:
import base64
from io import BytesIO
from PIL import Image

def base64_to_image(base64_string):
    # Decodificar a string Base64 para bytes
    image_bytes = base64.b64decode(base64_string)

    # Criar um buffer de memória e carregar a imagem com PIL
    image = Image.open(BytesIO(image_bytes))

    return image

# Exemplo de uso com uma string Base64 (substitua pela sua)
base64_string = event["image_data"]

# Converter e exibir a imagem
img = base64_to_image(base64_string)
img  # Exibe a imagem

In [ ]:
for k, v in result_dict.items():
  print(k, ":\n\t")
  print("Diagnóstico:\n\t", v['diagnosis'],'\n', 100*"=",'\n')
  print("Prognóstico:\n\t", v['prognosis'],'\n', 100*"=",'\n')
  print("Relatório:\n\t", v['validation'])

In [ ]:
event['im_teste'][1].squeeze(0).shape

In [ ]:
def convert_to_base64(pil_image):
  """
  Convert PIL images to Base64 encoded strings

  :param pil_image: PIL image
  :return: Base64 string
  """
  pil_image = pil_image.convert("RGB")
  buffered = BytesIO()
  pil_image.save(buffered, format="JPEG")  # You can change the format if needed
  img_str = base64.b64encode(buffered.getvalue()).decode("utf-8")
  return img_str

In [ ]:
import base64
from langchain_core.messages import HumanMessage


# # Caminho da imagem
image_path = "/content/Image/train-image/image/ISIC_8966407.jpg"

# # Converter a imagem para Base64
with open(image_path, "rb") as image_file:
    encoded_string = base64.b64encode(image_file.read()).decode("utf-8")
# encoded_string="iVBORw0KGgoAAAANSUhEUgAAAG0AAABmCAYAAADBPx+VAAAACXBIWXMAAAsTAAALEwEAmpwYAAAAAXNSR0IArs4c6QAAAARnQU1BAACxjwv8YQUAAA3VSURBVHgB7Z27r0zdG8fX743i1bi1ikMoFMQloXRpKFFIqI7LH4BEQ+NWIkjQuSWCRIEoULk0gsK1kCBI0IhrQVT7tz/7zZo888yz1r7MnDl7z5xvsjkzs2fP3uu71nNfa7lkAsm7d++Sffv2JbNmzUqcc8m0adOSzZs3Z+/XES4ZckAWJEGWPiCxjsQNLWmQsWjRIpMseaxcuTKpG/7HP27I8P79e7dq1ars/yL4/v27S0ejqwv+cUOGEGGpKHR37tzJCEpHV9tnT58+dXXCJDdECBE2Ojrqjh071hpNECjx4cMHVycM1Uhbv359B2F79+51586daxN/+pyRkRFXKyRDAqxEp4yMlDDzXG1NPnnyJKkThoK0VFd1ELZu3TrzXKxKfW7dMBQ6bcuWLW2v0VlHjx41z717927ba22U9APcw7Nnz1oGEPeL3m3p2mTAYYnFmMOMXybPPXv2bNIPpFZr1NHn4HMw0KRBjg9NuRw95s8PEcz/6DZELQd/09C9QGq5RsmSRybqkwHGjh07OsJSsYYm3ijPpyHzoiacg35MLdDSIS/O1yM778jOTwYUkKNHWUzUWaOsylE00MyI0fcnOwIdjvtNdW/HZwNLGg+sR1kMepSNJXmIwxBZiG8tDTpEZzKg0GItNsosY8USkxDhD0Rinuiko2gfL/RbiD2LZAjU9zKQJj8RDR0vJBR1/Phx9+PHj9Z7REF4nTZkxzX4LCXHrV271qXkBAPGfP/atWvu/PnzHe4C97F48eIsRLZ9+3a3f/9+87dwP1JxaF7/3r17ba+5l4EcaVo0lj3SBq5kGTJSQmLWMjgYNei2GPT1MuMqGTDEFHzeQSP2wi/jGnkmPJ/nhccs44jvDAxpVcxnq0F6eT8h4ni/iIWpR5lPyA6ETkNXoSukvpJAD3AsXLiwpZs49+fPn5ke4j10TqYvegSfn0OnafC+Tv9ooA/JPkgQysqQNBzagXY55nO/oa1F7qvIPWkRL12WRpMWUvpVDYmxAPehxWSe8ZEXL20sadYIozfmNch4QJPAfeJgW3rNsnzphBKNJM2KKODo1rVOMRYik5ETy3ix4qWNI81qAAirizgMIc+yhTytx0JWZuNI03qsrgWlGtwjoS9XwgUhWGyhUaRZZQNNIEwCiXD16tXcAHUs79co0vSD8rrJCIW98pzvxpAWyyo3HYwqS0+H0BjStClcZJT5coMm6D2LOF8TolGJtK9fvyZpyiC5ePFi9nc/oJU4eiEP0jVoAnHa9wyJycITMP78+eMeP37sXrx44d6+fdt6f82aNdkx1pg9e3Zb5W+RSRE+n+VjksQWifvVaTKFhn5O8my63K8Qabdv33b379/PiAP//vuvW7BggZszZ072/+TJk91YgkafPn166zXB1rQHFvouAWHq9z3SEevSUerqCn2/dDCeta2jxYbr69evk4MHDyY7d+7MjhMnTiTPnz9Pfv/+nfQT2ggpO2dMF8cghuoM7Ygj5iWCqRlGFml0QC/ftGmTmzt3rmsaKDsgBSPh0/8yPeLLBihLkOKJc0jp8H8vUzcxIA1k6QJ/c78tWEyj5P3o4u9+jywNPdJi5rAH9x0KHcl4Hg570eQp3+vHXGyrmEeigzQsQsjavXt38ujRo44LQuDDhw+TW7duRS1HGgMxhNXHgflaNTOsHyKvHK5Ijo2jbFjJBQK9YwFd6RVMzfgRBmEfP37suBBm/p49e1qjEP2mwTViNRo0VJWH1deMXcNK08uUjVUu7s/zRaL+oLNxz1bpANco4npUgX4G2eFbpDFyQoQxojBCpEGSytmOH8qrH5Q9vuzD6ofQylkCUmh8DBAr+q8JCyVNtWQIidKQE9wNtLSQnS4jDSsxNHogzFuQBw4cyM61UKVsjfr3ooBkPSqqQHesUPWVtzi9/vQi1T+rJj7WiTz4Pt/l3LxUkr5P2VYZaZ4URpsE+st/dujQoaBBYokbrz/8TJNQYLSonrPS9kUaSkPeZyj1AWSj+d+VBoy1pIWVNed8P0Ll/ee5HdGRhrHhR5GGN0r4LGZBaj8oFDJitBTJzIZgFcmU0Y8ytWMZMzJOaXUSrUs5RxKnrxmbb5YXO9VGUhtpXldhEUogFr3IzIsvlpmdosVcGVGXFWp2oU9kLFL3dEkSz6NHEY1sjSRdIuDFWEhd8KxFqsRi1uM/nz9/zpxnwlESONdg6dKlbsaMGS4EHFHtjFIDHwKOo46l4TxSuxgDzi+rE2jg+BaFruOX4HXa0Nnf1lwAPufZeF8/r6zD97WK2qFnGjBxTw5qNGPxT+5T/r7/7RawFC3j4vTp09koCxkeHjqbHJqArmH5UrFKKksnxrK7FuRIs8STfBZv+luugXZ2pR/pP9Ois4z+TiMzUUkUjD0iEi1fzX8GmXyuxUBRcaUfykV0YZnlJGKQpOiGB76x5GeWkWWJc3mOrK6S7xdND+W5N6XyaRgtWJFe13GkaZnKOsYqGdOVVVbGupsyA/l7emTLHi7vwTdirNEt0qxnzAvBFcnQF16xh/TMpUuXHDowhlA9vQVraQhkudRdzOnK+04ZSP3DUhVSP61YsaLtd/ks7ZgtPcXqPqEafHkdqa84X6aCeL7YWlv6edGFHb+ZFICPlljHhg0bKuk0CSvVznWsotRu433alNdFrqG45ejoaPCaUkWERpLXjzFL2Rpllp7PJU2a/v7Ab8N05/9t27Z16KUqoFGsxnI9EosS2niSYg9SpU6B4JgTrvVW1flt1sT+0ADIJU2maXzcUTraGCRaL1Wp9rUMk16PMom8QhruxzvZIegJjFU7LLCePfS8uaQdPny4jTTL0dbee5mYokQsXTIWNY46kuMbnt8Kmec+LGWtOVIl9cT1rCB0V8WqkjAsRwta93TbwNYoGKsUSChN44lgBNCoHLHzquYKrU6qZ8lolCIN0Rh6cP0Q3U6I6IXILYOQI513hJaSKAorFpuHXJNfVlpRtmYBk1Su1obZr5dnKAO+L10Hrj3WZW+E3qh6IszE37F6EB+68mGpvKm4eb9bFrlzrok7fvr0Kfv727dvWRmdVTJHw0qiiCUSZ6wCK+7XL/AcsgNyL74DQQ730sv78Su7+t/A36MdY0sW5o40ahslXr58aZ5HtZB8GH64m9EmMZ7FpYw4T6QnrZfgenrhFxaSiSGXtPnz57e9TkNZLvTjeqhr734CNtrK41L40sUQckmj1lGKQ0rC37x544r8eNXRpnVE3ZZY7zXo8NomiO0ZUCj2uHz58rbXoZ6gc0uA+F6ZeKS/jhRDUq8MKrTho9fEkihMmhxtBI1DxKFY9XLpVcSkfoi8JGnToZO5sU5aiDQIW716ddt7ZLYtMQlhECdBGXZZMWldY5BHm5xgAroWj4C0hbYkSc/jBmggIrXJWlZM6pSETsEPGqZOndr2uuuR5rF169a2HoHPdurUKZM4CO1WTPqaDaAd+GFGKdIQkxAn9RuEWcTRyN2KSUgiSgF5aWzPTeA/lN5rZubMmR2bE4SIC4nJoltgAV/dVefZm72AtctUCJU2CMJ327hxY9t7EHbkyJFseq+EJSY16RPo3Dkq1kkr7+q0bNmyDuLQcZBEPYmHVdOBiJyIlrRDq41YPWfXOxUysi5fvtyaj+2BpcnsUV/oSoEMOk2CQGlr4ckhBwaetBhjCwH0ZHtJROPJkyc7UjcYLDjmrH7ADTEBXFfOYmB0k9oYBOjJ8b4aOYSe7QkKcYhFlq3QYLQhSidNmtS2RATwy8YOM3EQJsUjKiaWZ+vZToUQgzhkHXudb/PW5YMHD9yZM2faPsMwoc7RciYJXbGuBqJ1UIGKKLv915jsvgtJxCZDubdXr165mzdvtr1Hz5LONA8jrUwKPqsmVesKa49S3Q4WxmRPUEYdTjgiUcfUwLx589ySJUva3oMkP6IYddq6HMS4o55xBJBUeRjzfa4Zdeg56QZ43LhxoyPo7Lf1kNt7oO8wWAbNwaYjIv5lhyS7kRf96dvm5Jah8vfvX3flyhX35cuX6HfzFHOToS1H4BenCaHvO8pr8iDuwoUL7tevX+b5ZdbBair0xkFIlFDlW4ZknEClsp/TzXyAKVOmmHWFVSbDNw1l1+4f90U6IY/q4V27dpnE9bJ+v87QEydjqx/UamVVPRG+mwkNTYN+9tjkwzEx+atCm/X9WvWtDtAb68Wy9LXa1UmvCDDIpPkyOQ5ZwSzJ4jMrvFcr0rSjOUh+GcT4LSg5ugkW1Io0/SCDQBojh0hPlaJdah+tkVYrnTZowP8iq1F1TgMBBauufyB33x1v+NWFYmT5KmppgHC+NkAgbmRkpD3yn9QIseXymoTQFGQmIOKTxiZIWpvAatenVqRVXf2nTrAWMsPnKrMZHz6bJq5jvce6QK8J1cQNgKxlJapMPdZSR64/UivS9NztpkVEdKcrs5alhhWP9NeqlfWopzhZScI6QxseegZRGeg5a8C3Re1Mfl1ScP36ddcUaMuv24iOJtz7sbUjTS4qBvKmstYJoUauiuD3k5qhyr7QdUHMeCgLa1Ear9NquemdXgmum4fvJ6w1lqsuDhNrg1qSpleJK7K3TF0Q2jSd94uSZ60kK1e3qyVpQK6PVWXp2/FC3mp6jBhKKOiY2h3gtUV64TWM6wDETRPLDfSakXmH3w8g9Jlug8ZtTt4kVF0kLUYYmCCtD/DrQ5YhMGbA9L3ucdjh0y8kOHW5gU/VEEmJTcL4Pz/f7mgoAbYkAAAAAElFTkSuQmCC"
# encoded_string=image_path

# msg = [{"role": "user",
#         # "content": "Descreva a imagem",
#         "content": "What is in this picture?",
#         # "content": "Descreva a imagem com base no checklist de 7 pontos para detecção de cancer de pele .",
#         "image": [encoded_string]
#         }]
# encoded_string = event['image_data']
# encoded_string = convert_to_base64(T.ToPILImage()(event['im_teste'][3][0]))
content_str = """
Chat, você conhece o algoritmo ABCD para identificação de melanoma maligno? Poderia descreve-lo para mim com base na imagem que estou te entregando?
"""
content = [
    {'type': 'image_url', 'image_url': f'data:image/jpeg;base64,{encoded_string}'},
    {'type': 'text', 'text': content_str}
]
msg = [HumanMessage(content=content)]
response = llm.invoke(msg)
print(response.content)

In [ ]:
!cp /content/Image/train-image/image/ISIC_9972649.jpg /content/ISIC_9972649.jpg

In [ ]:
dt_ref[dt_ref['target'] == 1]['isic_id']

In [ ]:
print(f"""
        Duas imagens foram fornecidas para análise:
          1️⃣ Imagem original da lesão de pele, que mantém seu contexto visual completo.
          2️⃣ Imagem segmentada, que realça os contornos e características internas da lesão.

        Por favor, observe as características da mancha ou pinta nas imagens fornecidas e retorne uma resposta no
        seguinte formato JSON, baseado nas observações visuais:
        Adicionalmente, o maior diâmetro da lesão é de {4.18} mm.
        {{
          "Pontos": {{
            "Assimetria": "Sim ou Não. Motivo",
            "Bordas Irregulares": "Sim ou Não. Motivo",
            "Cor Desigual": "Sim ou Não. Motivo",
            "Diâmetro Maior que 6 mm": "Sim ou Não. Motivo",

            "Sangramento ou Secreção": "Sim ou Não. Motivo"
          }},
          "Resposta Final": "Quantidade de 'Sim' nos pontos"
        }}
        Cada campo deve conter 'Sim' ou 'Não', seguido de uma explicação breve do motivo da observação (por exemplo, 'Sim, a mancha é assimétrica', ou 'Não, as bordas são regulares'). A resposta final será a contagem do número de 'Sim' nos pontos observados.
      """)